In [ ]:
!git clone https://github.com/tuannh1905/FloodEdgeNet.git
%cd FloodEdgeNet
!pip install -r requirements.txt


In [ ]:
import os

MODEL_NAME = "flood_edgenet"

AUTO_PRETRAINED_MODELS = {'deeplabv3'}
IS_PRETRAINED = MODEL_NAME.lower() in AUTO_PRETRAINED_MODELS
PRETRAINED_FLAG = "--pretrained" if IS_PRETRAINED else ""

IMAGE_SIZE = 128
BATCH_SIZE = 4
EPOCHS = 100
LR = 0.001
LOSS_NAME = "lovasz_focal"
NUM_CLASSES = 1

MULTISEED = True
SEEDS = [42, 123, 456, 789, 2024]
REP_SEED = 42

QAT_EPOCHS = 15
QAT_LR = 1e-4

def get_checkpoint_path(model, loss, dataset, seed, is_multiseed):
    clean_loss = loss.lower().replace('-', '_')
    candidates = [
        f"outputs/seed_{seed}/{model}_{clean_loss}_{dataset}_s{seed}.pth",
        f"outputs/{model}_{clean_loss}_{dataset}_s{seed}.pth",
        f"outputs/seed_{seed}/{model}_{loss}_{dataset}_s{seed}.pth",
        f"outputs/{model}_{loss}_{dataset}_s{seed}.pth"
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return candidates[0] if is_multiseed else candidates[1]


In [ ]:
if MULTISEED:
    seeds_str = " ".join(map(str, SEEDS))
    !python main.py --dataset floodkaggle --model {MODEL_NAME} --num_classes {NUM_CLASSES} --loss {LOSS_NAME} --size {IMAGE_SIZE} --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --multiseed --seeds {seeds_str} {PRETRAINED_FLAG} --download
else:
    !python main.py --dataset floodkaggle --model {MODEL_NAME} --num_classes {NUM_CLASSES} --loss {LOSS_NAME} --size {IMAGE_SIZE} --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --seed {REP_SEED} {PRETRAINED_FLAG} --download


In [ ]:
weights_fk = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "floodkaggle", REP_SEED, MULTISEED)
!python qat.py --weights {weights_fk} --model {MODEL_NAME} --dataset floodkaggle --num_classes {NUM_CLASSES} --loss {LOSS_NAME} --size {IMAGE_SIZE} --epochs {QAT_EPOCHS} --lr {QAT_LR} --optimizer sgd


In [ ]:
weights_fk = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "floodkaggle", REP_SEED, MULTISEED)
!python export.py --model {MODEL_NAME} --weights {weights_fk} --size {IMAGE_SIZE} --num_classes {NUM_CLASSES} --output outputs/{MODEL_NAME}_floodkaggle.onnx


In [ ]:
if MULTISEED:
    seeds_str = " ".join(map(str, SEEDS))
    !python main.py --dataset aifloodsense --model {MODEL_NAME} --num_classes {NUM_CLASSES} --loss {LOSS_NAME} --size {IMAGE_SIZE} --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --multiseed --seeds {seeds_str} {PRETRAINED_FLAG} --download
else:
    !python main.py --dataset aifloodsense --model {MODEL_NAME} --num_classes {NUM_CLASSES} --loss {LOSS_NAME} --size {IMAGE_SIZE} --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --seed {REP_SEED} {PRETRAINED_FLAG} --download


In [ ]:
weights_afs = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "aifloodsense", REP_SEED, MULTISEED)
!python qat.py --weights {weights_afs} --model {MODEL_NAME} --dataset aifloodsense --num_classes {NUM_CLASSES} --loss {LOSS_NAME} --size {IMAGE_SIZE} --epochs {QAT_EPOCHS} --lr {QAT_LR} --optimizer sgd


In [ ]:
weights_afs = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "aifloodsense", REP_SEED, MULTISEED)
!python export.py --model {MODEL_NAME} --weights {weights_afs} --size {IMAGE_SIZE} --num_classes {NUM_CLASSES} --output outputs/{MODEL_NAME}_aifloodsense.onnx


In [ ]:
weights_fk = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "floodkaggle", REP_SEED, MULTISEED)
!python cross_dataset_eval.py --model {MODEL_NAME} --weights {weights_fk} --source_dataset floodkaggle --target_datasets floodkaggle aifloodsense floodvn --size {IMAGE_SIZE} --num_classes {NUM_CLASSES} --download


In [ ]:
weights_afs = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "aifloodsense", REP_SEED, MULTISEED)
!python cross_dataset_eval.py --model {MODEL_NAME} --weights {weights_afs} --source_dataset aifloodsense --target_datasets aifloodsense floodkaggle floodvn --size {IMAGE_SIZE} --num_classes {NUM_CLASSES} --download


In [ ]:
import os
import shutil
from google.colab import files

bundle_dir = f"{MODEL_NAME}_All_Artifacts"
os.makedirs(bundle_dir, exist_ok=True)
output_dir = "outputs"

if os.path.exists(output_dir):
    for item in sorted(os.listdir(output_dir)):
        src = os.path.join(output_dir, item)
        dst = os.path.join(bundle_dir, item)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        elif os.path.isfile(src):
            shutil.copy2(src, dst)

zip_name = f"{MODEL_NAME}_All_Results.zip"
shutil.make_archive(f"{MODEL_NAME}_All_Results", "zip", bundle_dir)
try:
    files.download(zip_name)
except Exception:
    pass
